# Super-resolution runner notebook

This notebook is a thin orchestrator. The implementation lives in the adjacent Python package `sr_modular/`.

Place your in-memory image array in `IMAGE_ARRAY` with shape `[N, 256, 448, 3]` and values in `[0, 255]`, then run the cells below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

In [ ]:
# Only for when I run in Colab
os.chdir("path/to/wd")

print(f"The current WD is: {os.getcwd()}")

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Make sure the local package is importable when the notebook is opened from this folder.
sys.path.insert(0, str(Path('.').resolve()))

from sr_modular import config as cfg
from sr_modular.datasets import build_dataloaders, train_val_test_split_indices
from sr_modular.train_srresnet import train_multiple_srresnet
from sr_modular.train_srgan import train_multiple_srgan
from sr_modular.super_resolve import (
    build_metrics_table,
    plot_srgan_histories,
    plot_srresnet_histories,
    show_absolute_difference_heatmaps,
    show_comparison_strips,
)
from sr_modular.utils import infer_image_array_from_namespace, validate_image_array

print(f"Using device: {cfg.DEVICE}")
print(f"Artifacts will be saved under: {cfg.PROJECT_ROOT.resolve()}")

In [ ]:
# -----------------------------
# DATA LOADING
# -----------------------------
def merge_cached_npy_folders(
    cache_root,
    start_folder=1,
    end_folder=42
):
    """
    Load cached per-folder .npy files, concatenate them, and flatten
    (N_seq, T, H, W, C) -> (N_seq * T, H, W, C).

    Expected files:
      cache_root/00001.npy
      cache_root/00002.npy
      ...
    """
    arrays = []

    for i in range(start_folder, end_folder + 1):
        folder_name = f"{i:05d}"
        cache_path = os.path.join(cache_root, f"{folder_name}.npy")

        if not os.path.exists(cache_path):
            print(f"[SKIP] Missing cache: {cache_path}")
            continue

        arr = np.load(cache_path)
        print(f"[LOAD] {folder_name}.npy shape={arr.shape}")
        arrays.append(arr)

    if len(arrays) == 0:
        raise ValueError("No cached .npy files were loaded.")

    merged = np.concatenate(arrays, axis=0)
    print("\nMerged sequence data shape:", merged.shape)

    N_seq, T, H, W, C = merged.shape
    data = merged.reshape(N_seq * T, H, W, C).copy()
    print("Flattened frame data shape:", data.shape)

    del arrays
    del merged

    return data

IMAGE_ARRAY = merge_cached_npy_folders(
    cache_root="path/to/cached/npy/folders",
    start_folder=1,
    end_folder=10
)

assert IMAGE_ARRAY.ndim == 4 and IMAGE_ARRAY.shape[-1] == 3
assert IMAGE_ARRAY.dtype == np.uint8, f"Expected uint8 frame cache, got {IMAGE_ARRAY.dtype}"
print("Loaded frame-level SR data:", IMAGE_ARRAY.shape)


## Optional: override defaults

Edit these values before building dataloaders or training if needed.


In [ ]:
# some things I change often
# cfg.SRRESNET_LOSS_FUNCTIONS = ["MSE", "L1"]
cfg.SRRESNET_LOSS_FUNCTIONS = ["Huber"]
cfg.SRRESNET_EPOCHS = 20
cfg.SRGAN_EPOCHS = 20
cfg.TRAIN_FRACTION = 0.8
cfg.VAL_FRACTION = 0.1
cfg.TEST_FRACTION = 0.1

print("Current losses:", cfg.SRRESNET_LOSS_FUNCTIONS)
print("Current # of SRResNet Epochs:", cfg.SRRESNET_EPOCHS)
print("Current # of SRGAN Epochs:", cfg.SRGAN_EPOCHS)
print("Current split fractions:", cfg.TRAIN_FRACTION, cfg.VAL_FRACTION, cfg.TEST_FRACTION)

## Build train / val / test splits and dataloaders

In [ ]:
SPLIT_INDICES = train_val_test_split_indices(
    n=len(IMAGE_ARRAY),
    train_fraction=cfg.TRAIN_FRACTION,
    val_fraction=cfg.VAL_FRACTION,
    test_fraction=cfg.TEST_FRACTION,
    seed=cfg.SEED,
)

TRAIN_LOADER, VAL_LOADER, TEST_LOADER = build_dataloaders(
    image_array=IMAGE_ARRAY,
    split_indices=SPLIT_INDICES,
    crop_size=cfg.CROP_SIZE,
    scaling_factor=cfg.SCALING_FACTOR,
    train_batch_size=cfg.TRAIN_BATCH_SIZE,
    eval_batch_size=cfg.EVAL_BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
)

print(f"Total images: {len(IMAGE_ARRAY)}")
print(f"Train split:   {len(SPLIT_INDICES['train'])}")
print(f"Val split:     {len(SPLIT_INDICES['val'])}")
print(f"Test split:    {len(SPLIT_INDICES['test'])}")

## Train SRResNet variants

In [ ]:
RUN_SRRESNET_TRAINING = True

if RUN_SRRESNET_TRAINING:
    srresnet_results = train_multiple_srresnet(
        loss_names=cfg.SRRESNET_LOSS_FUNCTIONS,
        train_loader=TRAIN_LOADER,
        val_loader=VAL_LOADER,
        num_epochs=cfg.SRRESNET_EPOCHS,
        lr=cfg.SRRESNET_LR,
    )
else:
    srresnet_results = {}
    print("SRResNet training skipped. Set RUN_SRRESNET_TRAINING = True to train.")

## Train SRGAN variants from the matching SRResNet checkpoints

In [ ]:
RUN_SRGAN_TRAINING = True

if RUN_SRGAN_TRAINING:
    srgan_results = train_multiple_srgan(
        loss_names=cfg.SRRESNET_LOSS_FUNCTIONS,
        train_loader=TRAIN_LOADER,
        val_loader=VAL_LOADER,
        num_epochs=cfg.SRGAN_EPOCHS,
        lr=cfg.SRGAN_LR,
    )
else:
    srgan_results = {}
    print("SRGAN training skipped. Set RUN_SRGAN_TRAINING = True to train.")

## Comparisons, visualizations, and summary table

## del later

In [ ]:
def show_srgan_loss_grid(
    test_loader,
    sample_index=0,
    save_path=None,
    figsize=(8.5, 11),
    dpi=300,
    show=True,
):
    """
    Display a 4 x 2 thesis-sized figure using SRGAN outputs loaded
    from the saved model checkpoint paths.

    Layout:
        HR Ground Truth        MSE Loss
        L1 Loss                G-Loss
        SSIM Loss              MS-SSIM Loss
        MS-SSIM + L1 Loss      Huber Loss

    Assumes the following are already imported into the notebook
    from super_resolve.py / your project:
        - cfg
        - discover_best_model_paths
        - load_srgan_generator_from_checkpoint
        - tensor_to_uint8
        - torch
        - plt
        - np

    Parameters
    ----------
    test_loader : DataLoader
        Test dataloader.
    sample_index : int
        Index of the sample from test_loader.dataset.
    save_path : str or Path or None
        Optional file path to save the figure.
    figsize : tuple
        Figure size in inches.
    dpi : int
        DPI for saving/display.
    show : bool
        Whether to display the figure.

    Returns
    -------
    fig, axes
    """

    def canonical_name(name):
        return (
            str(name)
            .upper()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
            .replace("+", "")
            .replace("PLUS", "")
            .replace("LOSS", "")
        )

    def resolve_srgan_key(srgan_paths, candidates, label):
        """
        Match a desired loss label to one of the discovered SRGAN checkpoint keys.
        """
        available_keys = list(srgan_paths.keys())

        # exact match first
        for cand in candidates:
            cand_upper = cand.upper()
            if cand_upper in srgan_paths:
                return cand_upper

        # canonical fuzzy match second
        candidate_norms = {canonical_name(c) for c in candidates}
        for key in available_keys:
            if canonical_name(key) in candidate_norms:
                return key

        raise KeyError(
            f"Could not resolve SRGAN checkpoint for '{label}'. "
            f"Available SRGAN keys: {available_keys}"
        )

    plot_spec = [
        ("HR Ground Truth", None),
        ("MSE Loss", ["MSE"]),
        ("L1 Loss", ["L1"]),
        ("G-Loss", ["G_LOSS", "GLOSS", "G"]),
        ("SSIM Loss", ["SSIM"]),
        ("MS-SSIM Loss", ["MS_SSIM", "MSSSIM"]),
        ("MS-SSIM + L1 Loss", ["MS_SSIM_L1", "MSSSIML1", "MS_SSIM_PLUS_L1"]),
        ("Huber Loss", ["HUBER"]),
    ]

    # Only use saved model paths discovered from your project.
    _, srgan_paths = discover_best_model_paths()

    if not srgan_paths:
        raise FileNotFoundError("No SRGAN checkpoints were found by discover_best_model_paths().")

    # Get the sample directly from the dataset.
    lr_img_imagenet, hr_img_minus1_to1 = test_loader.dataset[sample_index]

    lr_batch = lr_img_imagenet.unsqueeze(0).to(cfg.DEVICE)
    hr_uint8 = tensor_to_uint8(hr_img_minus1_to1.cpu(), source="[-1, 1]")

    images = {"HR Ground Truth": hr_uint8}
    used_checkpoints = {}

    # Generate each SRGAN image from its saved checkpoint path.
    for label, candidates in plot_spec:
        if candidates is None:
            continue

        resolved_key = resolve_srgan_key(srgan_paths, candidates, label)
        checkpoint_path = srgan_paths[resolved_key]
        used_checkpoints[label] = checkpoint_path

        generator = load_srgan_generator_from_checkpoint(checkpoint_path)
        sr_out = generator(lr_batch).squeeze(0).cpu()
        sr_uint8 = tensor_to_uint8(sr_out, source="[-1, 1]")

        images[label] = sr_uint8

    # Plot 4 x 2 layout.
    fig, axes = plt.subplots(4, 2, figsize=figsize, dpi=dpi)
    axes = axes.ravel()

    for ax, (label, _) in zip(axes, plot_spec):
        ax.imshow(images[label])
        ax.set_title(label, fontsize=12)
        ax.axis("off")

    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")

    if show:
        plt.show()

    print("SRGAN checkpoints used:")
    for label, path in used_checkpoints.items():
        print(f"  {label}: {path}")

    return fig, axes

In [ ]:
from sr_modular.super_resolve import *

In [ ]:
def plot_srgan_content_loss_curves(
    loss_names=None,
    figsize=(10, 5),
    dpi=300,
    save_dir=None,
    show=True,
):
    """
    Plot SRGAN content loss curves as two separate figures:
        1. Training content loss
        2. Validation content loss

    Assumes the following are already imported from super_resolve.py:
        - cfg
        - get_history_if_present
        - plt
        - Path, if save_dir is used

    Parameters
    ----------
    loss_names : list[str] or None
        Loss names to plot. If None, uses cfg.SRRESNET_LOSS_FUNCTIONS.

    figsize : tuple
        Figure size for each plot.

    dpi : int
        Figure DPI.

    save_dir : str, Path, or None
        Optional directory to save the figures.

    show : bool
        Whether to display the figures.

    Returns
    -------
    fig_train, ax_train, fig_val, ax_val
    """

    def display_label(loss_name):
        suffix = loss_name.upper()

        if suffix in {"G", "GLOSS", "G_LOSS", "G-LOSS"}:
            return "G-LOSS"

        return suffix

    loss_names = cfg.SRRESNET_LOSS_FUNCTIONS if loss_names is None else loss_names

    histories = {}

    for loss_name in loss_names:
        suffix = loss_name.upper()
        model_name = f"SRGAN_ResNet_{suffix}"
        hist = get_history_if_present(model_name)

        if hist is not None:
            histories[suffix] = hist

    if not histories:
        raise FileNotFoundError("No SRGAN history JSON files found.")

    # -------------------------
    # Training content loss plot
    # -------------------------
    fig_train, ax_train = plt.subplots(figsize=figsize, dpi=dpi)

    for suffix, hist in histories.items():
        if "train_content_loss" not in hist:
            raise KeyError(
                f"History for {suffix} does not contain 'train_content_loss'. "
                f"Available keys: {list(hist.keys())}"
            )

        ax_train.plot(
            hist["train_content_loss"],
            label=f"{display_label(suffix)} train",
        )

    ax_train.set_title("SRGAN: Train Content Loss")
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("Content Loss")
    ax_train.legend()
    fig_train.tight_layout()

    # ---------------------------
    # Validation content loss plot
    # ---------------------------
    fig_val, ax_val = plt.subplots(figsize=figsize, dpi=dpi)

    for suffix, hist in histories.items():
        if "val_content_loss" not in hist:
            raise KeyError(
                f"History for {suffix} does not contain 'val_content_loss'. "
                f"Available keys: {list(hist.keys())}"
            )

        ax_val.plot(
            hist["val_content_loss"],
            label=f"{display_label(suffix)} val",
        )

    ax_val.set_title("SRGAN: Validation Content Loss")
    ax_val.set_xlabel("Epoch")
    ax_val.set_ylabel("Content Loss")
    ax_val.legend()
    fig_val.tight_layout()

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        train_path = save_dir / "srgan_train_content_loss.pdf"
        val_path = save_dir / "srgan_validation_content_loss.pdf"

        fig_train.savefig(train_path, dpi=dpi, bbox_inches="tight")
        fig_val.savefig(val_path, dpi=dpi, bbox_inches="tight")

        print(f"Saved train content loss plot to: {train_path}")
        print(f"Saved validation content loss plot to: {val_path}")

    if show:
        plt.show()

    return fig_train, ax_train, fig_val, ax_val

In [ ]:
fig_train, ax_train, fig_val, ax_val = plot_srgan_content_loss_curves(
    #save_dir="figures"
)

In [ ]:
def show_image_from_array(
    index,
    image_array=IMAGE_ARRAY,
    title=None,
    figsize=(6, 6),
    cmap=None,
    axis_off=True,
):
    """
    Display one image from IMAGE_ARRAY by index.

    Handles common image formats:
        - NumPy arrays
        - Torch tensors
        - H x W
        - H x W x C
        - C x H x W
    """

    img = image_array[index]

    # Convert torch tensor to numpy if needed
    if hasattr(img, "detach"):
        img = img.detach().cpu().numpy()

    # Remove singleton dimensions
    img = np.squeeze(img)

    # Convert C x H x W to H x W x C if needed
    if img.ndim == 3 and img.shape[0] in [1, 3, 4]:
        img = np.transpose(img, (1, 2, 0))

    # If values are in [-1, 1], convert to [0, 1]
    if img.min() < 0:
        img = (img + 1.0) / 2.0

    # If values are in [0, 255], convert to [0, 1] for display
    if img.max() > 1.0:
        img = img / 255.0

    img = np.clip(img, 0.0, 1.0)

    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)

    if title is None:
        title = f"IMAGE_ARRAY[{index}]"

    plt.title(title)

    if axis_off:
        plt.axis("off")

    plt.show()

In [ ]:
show_image_from_array(3457, title='Frame 3')

In [ ]:
show_image_from_array(3456, title='Frame 2')

In [ ]:
show_image_from_array(3455, title='Frame 1')

In [ ]:
fig, axes = show_srgan_loss_grid(
    TEST_LOADER,
    sample_index=178
)

In [ ]:
RUN_COMPARISONS = True

if RUN_COMPARISONS:
    plot_srresnet_histories(loss_names=cfg.SRRESNET_LOSS_FUNCTIONS)
    plot_srgan_histories(loss_names=cfg.SRRESNET_LOSS_FUNCTIONS)
    show_comparison_strips(test_loader=TEST_LOADER, loss_names=cfg.SRRESNET_LOSS_FUNCTIONS, sample_index=0)
    show_absolute_difference_heatmaps(test_loader=TEST_LOADER, loss_names=cfg.SRRESNET_LOSS_FUNCTIONS, sample_index=0)

    summary_df = build_metrics_table(
        val_loader=VAL_LOADER,
        test_loader=TEST_LOADER,
        loss_names=cfg.SRRESNET_LOSS_FUNCTIONS,
    )
    display(summary_df)
else:
    summary_df = None
    print("Comparisons skipped. Set RUN_COMPARISONS = True to generate plots and the summary table.")

In [ ]:
for i_ in range(1,20):
  show_comparison_strips(test_loader=TEST_LOADER, loss_names=cfg.SRRESNET_LOSS_FUNCTIONS, sample_index=i_)
  show_absolute_difference_heatmaps(test_loader=TEST_LOADER, loss_names=cfg.SRRESNET_LOSS_FUNCTIONS, sample_index=i_)

In [ ]:
summary_df.sort_values(by = "test_ssim", ascending=False)

In [ ]:
if summary_df is not None:
    summary_path = cfg.METRICS_DIR / "model_metrics_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved summary table to: {summary_path}")